# FloodWay — Phase 4B: Terrain Feature Extraction (Google Colab)
### Extracting 30m Elevation & Slope without Google Earth Engine

This notebook performs reproducible terrain feature extraction for the 40-record FloodWay pilot cohort.

---
### 1. Data Source & Provenance
* **Dataset:** Copernicus DEM GLO-30 (Global 30-meter Digital Surface Model)
* **Provider:** European Space Agency (ESA) / Airbus, hosted on **AWS Open Data** as Cloud Optimized GeoTIFFs (COGs)
* **Access Requirements:** **100% Free, NO API key, NO AWS account, NO authentication.** Open to public HTTP range requests.
* **Coordinate Reference System (CRS):** WGS84 (`EPSG:4326`)
* **Spatial Resolution:** 1 arc-second (~30.87 meters)

> **Important DSM Caveat:** Copernicus GLO-30 is a **Digital Surface Model (DSM)**, not a bare-earth Digital Terrain Model (DTM). Elevation measurements reflect the top of the canopy in tropical rainforests and oil palm plantations, as well as building heights in urban areas.

---
### 2. Geospatial Methodology
1. **Windowed HTTP Range Requests:** Instead of downloading 44 MB full tiles, `rasterio` queries only a $3 \times 3$ pixel patch (~$90\text{ m} \times 90\text{ m}$) centered on each point via HTTP range headers (~2 KB transfer per point).
2. **Horn's Slope Calculation:** Local slope in degrees ($0^\circ - 90^\circ$) is derived across the $3 \times 3$ neighborhood using Horn's finite-difference algorithm, with longitudinal distances dynamically scaled by $\cos(\text{latitude})$ to preserve true ground geometry.
3. **Zero-Imputation Policy:** Missing values or coastal NoData pixels are strictly preserved as `NaN` and flagged with `missing_terrain_flag = True`. No silent substitution with $0$ occurs.
4. **No Constant TWI:** Arbitrary constant-catchment approximations of Topographic Wetness Index are completely excluded.

## Step 1: Install Dependencies
Install `rasterio` for Cloud Optimized GeoTIFF (COG) window streaming.

In [ ]:
# Install rasterio with embedded GDAL
!pip install -q rasterio pandas numpy

## Step 2: Upload `floodway_pilot_rainfall_40.csv`
Upload the Phase 4A enriched rainfall dataset from your local machine. If running in a repository environment, it will load directly.

In [ ]:
import os
import pandas as pd

input_filename = "floodway_pilot_rainfall_40.csv"
if not os.path.exists(input_filename) and os.path.exists(f"datasets/{input_filename}"):
    input_filename = f"datasets/{input_filename}"

if not os.path.exists(input_filename):
    print(f"Please upload {input_filename} using the file browser on the left, or run the file uploader below:")
    try:
        from google.colab import files
        uploaded = files.upload()
        input_filename = "floodway_pilot_rainfall_40.csv"
    except Exception as e:
        print("File upload prompt skipped.")

df = pd.read_csv(input_filename)
print(f"Successfully loaded {len(df)} records from {input_filename}")
df.head(3)

## Step 3: Define COG Window Streaming & Horn's Slope Algorithm

In [ ]:
import math
import time
import numpy as np
import rasterio
from rasterio.windows import Window

def calculate_horn_slope(patch, lat_deg):
    """
    Computes terrain slope in degrees using Horn (1981) finite-difference algorithm
    across a 3x3 pixel neighborhood, scaled by local geodesic ground distance.
    """
    z_tl, z_t, z_tr = patch[0, 0], patch[0, 1], patch[0, 2]
    z_l,  z_c, z_r  = patch[1, 0], patch[1, 1], patch[1, 2]
    z_bl, z_b, z_br = patch[2, 0], patch[2, 1], patch[2, 2]

    # 1 arc-second ground distance in meters
    dy = 30.867  # ~111,120 m / 3600
    dx = 30.867 * math.cos(math.radians(lat_deg))

    # Horn's weighted gradient
    dz_dx = ((z_tr + 2.0 * z_r + z_br) - (z_tl + 2.0 * z_l + z_bl)) / (8.0 * dx)
    dz_dy = ((z_tl + 2.0 * z_t + z_tr) - (z_bl + 2.0 * z_b + z_br)) / (8.0 * dy)

    slope_rad = math.atan(math.sqrt(dz_dx**2 + dz_dy**2))
    return math.degrees(slope_rad)

def extract_point_terrain(lat, lon, tile_cache):
    """
    Streams a 3x3 window from AWS Copernicus DEM GLO-30 COG via HTTP byte-range requests.
    """
    northing = f"N{int(math.floor(lat)):02d}_00"
    easting = f"E{int(math.floor(lon)):03d}_00"
    tile_name = f"Copernicus_DSM_COG_10_{northing}_{easting}_DEM"
    cog_url = f"https://copernicus-dem-30m.s3.amazonaws.com/{tile_name}/{tile_name}.tif"

    try:
        if cog_url not in tile_cache:
            tile_cache[cog_url] = rasterio.open(cog_url)
        src = tile_cache[cog_url]

        py, px = src.index(lon, lat)
        if px < 1 or px >= src.width - 1 or py < 1 or py >= src.height - 1:
            return np.nan, np.nan, "COORDINATE_ON_TILE_EDGE"

        win = Window(px - 1, py - 1, 3, 3)
        patch = src.read(1, window=win)

        if src.nodata is not None and np.any(patch == src.nodata):
            return np.nan, np.nan, "CONTAINS_NODATA_PIXEL"
        if np.any(np.isnan(patch)) or np.any(patch < -500.0) or np.any(patch > 9000.0):
            return np.nan, np.nan, "INVALID_ELEVATION_VALUE"

        elev = float(patch[1, 1])
        slope = calculate_horn_slope(patch, lat)
        return round(elev, 2), round(slope, 2), "VALID_COPERNICUS_30M"
    except Exception as e:
        return np.nan, np.nan, f"HTTP_RANGE_READ_ERROR: {str(e)}"


## Step 4: Execute Terrain Extraction across Cohort

In [ ]:
elevations = []
slopes = []
qc_statuses = []
missing_flags = []
tile_cache = {}

total = len(df)
print(f"Starting terrain extraction for {total} coordinates...")
start_time = time.time()

for idx, row in df.iterrows():
    lat, lon = float(row["latitude"]), float(row["longitude"])
    elev, slope, qc = extract_point_terrain(lat, lon, tile_cache)
    
    is_missing = np.isnan(elev) or np.isnan(slope)
    missing_flags.append(is_missing)
    elevations.append(elev)
    slopes.append(slope)
    qc_statuses.append(qc)
    
    print(f"[{idx+1:02d}/{total}] {row['sample_id']} ({row['state']}) -> Elev: {elev:6.2f} m | Slope: {slope:5.2f} deg | {qc}")

for src in tile_cache.values():
    src.close()

elapsed = time.time() - start_time
print(f"\nCompleted in {elapsed:.2f} seconds.")

# Enrich DataFrame
df["elevation_m"] = elevations
df["slope_deg"] = slopes
df["dem_source"] = "AWS_COPERNICUS_GLO30_COG"
df["dem_product_type"] = "DSM_SURFACE_MODEL"
df["dem_spatial_res_m"] = 30.0
df["dem_crs"] = "EPSG:4326"
df["missing_terrain_flag"] = missing_flags
df["terrain_qc_status"] = qc_statuses


## Step 5: Quality Assurance & Feature Distribution Analysis

In [ ]:
print("=" * 70)
print("PHASE 4B TERRAIN FEATURE EXTRACTION QA REPORT")
print("=" * 70)
valid_count = sum(not m for m in df["missing_terrain_flag"])
missing_count = sum(df["missing_terrain_flag"])
print(f"Total Records:      {len(df)}")
print(f"Valid Extractions:  {valid_count} ({valid_count/len(df)*100:.1f}%)")
print(f"Missing / Masked:   {missing_count}")
print("-" * 70)

for group_name, target_val in [("POSITIVES (Flood Events)", 1), ("PSEUDO-NEGATIVES (Background)", 0)]:
    gdf = df[df["target"] == target_val]
    print(f"\n--- {group_name} (N = {len(gdf)}) ---")
    for col in ["elevation_m", "slope_deg", "rain_3d_prior_mm", "rain_7d_prior_mm"]:
        vals = gdf[col].dropna()
        if len(vals) > 0:
            print(f"  {col:<16}: Mean = {vals.mean():6.2f} | Median = {vals.median():6.2f} | IQR = [{vals.quantile(0.25):5.1f}, {vals.quantile(0.75):5.1f}] | Min = {vals.min():5.1f} | Max = {vals.max():5.1f}")
print("=" * 70)


## Step 6: Export Output Dataset
Save the unified environmental dataset containing rainfall, elevation, and slope to `floodway_pilot_features_40.csv`.

In [ ]:
out_filename = "floodway_pilot_features_40.csv"
df.to_csv(out_filename, index=False)
print(f"Successfully saved {len(df)} records to {out_filename}")

# Download file to local machine (if running in Colab)
try:
    from google.colab import files
    files.download(out_filename)
    print(f"Download prompt opened for {out_filename}")
except Exception:
    pass
